In [1]:
from pathlib import Path

import pandas as pd

MAX_REASONING_CHARS = 8192
DEFAULT_TAIL_CHARS = 1600

ORIGINAL_TRAIN_PATH = Path("../../data/out/splits/random/mmlu/train_original.parquet")
DISTILL_PATH = Path("../../data/out/distillation/mmlu_corrected_answer_deepseek_v4_pro_and_others.parquet")
OUT_DIR = Path("../../data/out/splits/random/mmlu/")
OUT_NAME_HEAD = f"train_corrected_answer_deepseek_v4_pro_and_others_head_truncated{MAX_REASONING_CHARS}.parquet"
OUT_NAME_MIDDLE = f"train_corrected_answer_deepseek_v4_pro_and_others_middle_truncated{MAX_REASONING_CHARS}.parquet"


In [2]:
train_ids = set(pd.read_parquet(ORIGINAL_TRAIN_PATH)["question_id"])
distill_df = pd.read_parquet(DISTILL_PATH)
print(f"Distill rows: {len(distill_df)}, original train ids: {len(train_ids)}")

train_df = distill_df[distill_df["question_id"].isin(train_ids)].reset_index(drop=True)
assert len(train_df) == len(train_ids), f"Expected {len(train_ids)} rows, got {len(train_df)}"
print(f"Filtered train rows: {len(train_df)}")

Distill rows: 12032, original train ids: 9626
Filtered train rows: 9626


In [3]:
def middle_truncate(
    reasoning,
    corrected,
    max_chars=MAX_REASONING_CHARS,
    default_tail=DEFAULT_TAIL_CHARS,
):
    if not isinstance(reasoning, str) or len(reasoning) <= max_chars:
        return reasoning

    has_corrected = isinstance(corrected, str) and corrected.strip() != ""
    if not has_corrected:
        return reasoning[:max_chars]

    head = len(reasoning) - len(corrected)
    original_reasoning = reasoning[:head]

    corrected_budget = max_chars - len(original_reasoning)
    if corrected_budget > default_tail:
        corrected_budget = default_tail

    corrected_reasoning = corrected[:corrected_budget]

    if not original_reasoning.endswith("\n"):
        original_reasoning += "\n"

    return original_reasoning + corrected_reasoning

In [4]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

too_long = int((train_df["distill_reasoning"].str.len() > MAX_REASONING_CHARS).sum())

# Variant 1: head truncation (first MAX_REASONING_CHARS chars)
head_df = train_df.copy()
head_df["distill_reasoning"] = head_df["distill_reasoning"].str.slice(0, MAX_REASONING_CHARS)
head_df.to_parquet(str(OUT_DIR / OUT_NAME_HEAD), index=False)

# Variant 2: middle drop (head + tail, tail sized by corrected_reasoning).
# If corrected_reasoning is absent, fall back to empty -> head 18000 + tail 6000.
middle_df = train_df.copy()
corrected = (
    middle_df["corrected_reasoning"]
    if "corrected_reasoning" in middle_df.columns
    else pd.Series([None] * len(middle_df), index=middle_df.index)
)
middle_df["distill_reasoning"] = [middle_truncate(r, c) for r, c in zip(middle_df["distill_reasoning"], corrected)]
middle_df.to_parquet(str(OUT_DIR / OUT_NAME_MIDDLE), index=False)

print(f"Truncated distill_reasoning in {too_long} rows (budget {MAX_REASONING_CHARS} chars)")
print(f"Saved head variant   -> {(OUT_DIR / OUT_NAME_HEAD).resolve()}")
print(f"Saved middle variant -> {(OUT_DIR / OUT_NAME_MIDDLE).resolve()}")

Truncated distill_reasoning in 1921 rows (budget 8192 chars)
Saved head variant   -> /Users/aigoncharov/dev/sktech/recursive_caft/data/out/splits/random/mmlu/train_corrected_answer_deepseek_v4_pro_and_others_head_truncated8192.parquet
Saved middle variant -> /Users/aigoncharov/dev/sktech/recursive_caft/data/out/splits/random/mmlu/train_corrected_answer_deepseek_v4_pro_and_others_middle_truncated8192.parquet
